In [0]:
%pip install xgboost

In [0]:
%restart_python
# dbutils.library.restartPython()

In [0]:
import pandas as pd
import seaborn as sns

import sklearn
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

import mlflow
from mlflow.models.signature import infer_signature

import logging
import json 
import os
from sys import version_info

import xgboost

In [0]:
logging.getLogger("mlflow").setLevel(logging.FATAL)

In [0]:
diamonds_df = sns.load_dataset('diamonds').drop(['cut', 'color', 'clarity'], axis=1)

X_train, X_test, y_train, y_test = train_test_split(diamonds_df.drop(["price"], axis=1), diamonds_df["price"], random_state=42)

In [0]:
class xgboost_regressor(mlflow.pyfunc.PythonModel):

    def __init__(self, params):
        """ Initialize with just the model hyperparameters """
        #
        self.params = params
        self.xgb_model = None
        self.config = None
        
    def load_context(self, context=None, config_path=None):
        """ When loading a pyfunc, this method runs automatically with the related
            context. This method is designed to perform the same functionality when
            run in a notebook or a downstream operation (like a REST endpoint).
            If the `context` object is provided, it will load the path to a config from 
            that object (this happens with `mlflow.pyfunc.load_model()` is called).
            If the `config_path` argument is provided instead, it uses this argument
            in order to load in the config. """
        #
        if context: # This block executes for server run
            config_path = context.artifacts["config_path"]
        else:       # This block executes for notebook run
            pass

        self.config = json.load(open(config_path))
      
    def preprocess_input(self, model_input):
        """ Return pre-processed model_input """
        #
        # any preprocessing can be done there. For the example purpose, let's here apply a Standard Scaler
        from sklearn.preprocessing import StandardScaler
        #
        for c in list(model_input.columns):
            model_input[c] = StandardScaler().fit_transform(model_input[[c]])
        #
        return model_input
  
    def fit(self, X_train, y_train):
        """ Uses the same preprocessing logic to fit the model """
        #
        from xgboost import XGBRegressor
        #
        processed_model_input = self.preprocess_input(X_train)
        xgb_model = XGBRegressor(**self.params)
        xgb_model.fit(processed_model_input, y_train)
        #
        self.xgb_model = xgb_model
    
    def predict(self, context, model_input):
        """ This is the main entrance to the model in deployment systems """
        #
        processed_model_input = self.preprocess_input(model_input.copy())
        return self.xgb_model.predict(processed_model_input)

#### Definition of the parameters for the custom xgboost model

In [0]:
params_xgb = {
    "n_estimators": 1000, 
    "max_depth": 7,
    "eta": 0.1,
    "subsample": 0.7,
    "colsample_bytree": 0.8
}

# Designate a path
config_path_xgb = "data_xgb.json"

# Save the results
with open(config_path_xgb, "w") as f:
    json.dump(params_xgb, f)

# Generate an artifact object to saved
# All paths to the associated values will be copied over when saving
artifacts_xgb = {"config_path": config_path_xgb} 

#### Instantiate the xgboost custom model

In [0]:
model_xgb = xgboost_regressor(params_xgb)
#
model_xgb.load_context(config_path=config_path_xgb) 
#
# Confirm the config has loaded
model_xgb.config

In [0]:
model_xgb.fit(X_train, y_train)

In [0]:
predictions_xgb = model_xgb.predict(context=None, model_input=X_test)
pd.DataFrame({'actual prices': list(y_test), 'predictions': list(predictions_xgb)}).head(5)

In [0]:
# get model signature
signature_xgb = infer_signature(X_test, predictions_xgb)
signature_xgb

#### Generate the conda environment. This can be arbitrarily complex. This is necessary because when we use mlflow.sklearn, we automatically log the appropriate version of sklearn. With a pyfunc, we must manually construct our deployment environment


In [0]:
conda_env_xgb = {
    "channels": ["defaults"],
    "dependencies": [
        f"python={version_info.major}.{version_info.minor}.{version_info.micro}",
        "pip",
        {"pip": ["mlflow",
                 f"xgboost=={xgboost.__version__}"]
        },
    ],
    "name": "xgboost_env"
}
conda_env_xgb

In [0]:
with mlflow.start_run() as run:
    mlflow.pyfunc.log_model(
        "xgb_regressor", 
        python_model=model_xgb, 
        artifacts=artifacts_xgb,
        conda_env=conda_env_xgb,
        signature=signature_xgb,
        input_example=X_test[:3] 
  )

In [0]:
mlflow_pyfunc_model_path_xgb = f"runs:/{run.info.run_id}/xgb_regressor"
loaded_preprocess_model_xgb = mlflow.pyfunc.load_model(mlflow_pyfunc_model_path_xgb)
#
y_pred_xgb = loaded_preprocess_model_xgb.predict(X_test)
#
pd.DataFrame({'actual prices': list(y_test), 'predictions': list(y_pred_xgb)}).head(5)

In [0]:
print("RMSE for custom xgboost model: ", mean_squared_error(y_test, y_pred_xgb, squared=False))

In [0]:
xgboost_custom_predict = mlflow.pyfunc.spark_udf(spark, mlflow_pyfunc_model_path_xgb)
display(spark.createDataFrame(X_test).withColumn('prediction', xgboost_custom_predict(*['carat', 'depth', 'table', 'x', 'y', 'z'])).limit(5))